# Rapor: Unmasked Padding Drift

Bu çalışma, padding (dolgu) ve masking işlemlerinin model davranışını nasıl etkilediğini incelemektedir. Özellikle kısa metinlerde eklenen padding tokenları, modelin dikkat ve temsil öğrenme mekanizmasını nasıl bozar veya yönlendirir, araştırılmıştır.

## Amaç
- Dolgu tokenları modelin kararını etkiler mi?
- Maskelenmemiş veya yanlış maskelenmiş girdiler modelin güvenini nasıl değiştirir?
- Kısa örneklerde pozisyonel drift ve padding etkisi belirginleşir mi?

## Gözlem
Unmasked padding drift, metinlerin sonuna eklenen padding alanlarının model tarafından anlam taşıyan bir bilgi gibi yorumlanması veya kontrolsüz şekilde etkilemesi durumu olarak tanımlanabilir. Özellikle maskelenmeyen veya yanlış maskelenen padding unsurları, modelin bağlamı bozan ve kararı etkileyen gereksiz sinyaller üretir.

Bu etkiler model mimarisine göre değişir. DistilBERT gibi daha küçük ve distile edilmiş modeller, baskın duygu sinyaline daha hızlı yapışma eğilimindedir; bu nedenle padding kaynaklı sapmalar kararlarını daha sert etkileyebilir. ALBERT ise katmanlar arası ağırlık paylaşımı nedeniyle gürültünün bir katmandan diğerine daha fazla yayılmasına neden olabilir. Bu durum, özellikle kısa ve nötr cümlelerde kararın daha dengesiz hale gelmesine yol açar.

## Neden Önemli?
Kısa metinlerde veri boyutları küçük olduğundan, padding’in etkisi daha baskın hale gelir. Bu durum, modelin çıkarımında “gerçek metin” ile “ekstra boşluk/işaret” ayrımının net olmadığı bir davranış yaratabilir.

## Sonuç
Padding ve maskelenme mekanizması, modelin performansını ve güvenilirliğini doğrudan etkileyebilen kritik bir bileşendir. Özellikle kısa örneklerde, ekleme ve boşluk işlemleri karar üzerinde belirleyici olabilir. Çalışma, modelin ne kadar güçlü olursa olsun özelikle padding maskelenmediğinde aynı cümlenin kararının mimariye göre farklı biçimde değişebileceğini göstermektedir.

## Değerlendirme
- Padding tokenlarının maske edilmesi ve güvenilir şekilde işlenmesi gerekir.
- Kısa metinlerde model çıktıları padding etkisi açısından ayrı değerlendirilmelidir.
- Padding kaynaklı drift için çoklu örnekli testler ve karşılaştırmalar yapılarak kararlar doğrulanmalıdır.
- Farklı mimariler arasında davranış farkı incelenmeli; özellikle DistilBERT ve ALBERT’in padding etkisi altında nasıl farklı tepki verdikleri değerlendirilmelidir.

Bu rapor, görünen metinlerden çok daha az fark edilen dolgu unsurlarının bile model davranışını değiştirebileceğini göstermektedir. Özellikle DistilBERT’in daha agresif duygu eğilimleri ve ALBERT’in katmanlar arası paylaşım nedeniyle gürültü büyütmesi, padding drift’in yapısal olarak model mimarisinden nasıl etkilendiğini ortaya koymaktadır.

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

checkpoints = {
    "DistilBERT (WordPiece)": "distilbert-base-uncased-finetuned-sst-2-english",
    "RoBERTa (BPE)": "textattack/roberta-base-SST-2",
    "ALBERT (SentencePiece)": "textattack/albert-base-v2-SST-2"
}

loaded_models = {}

for name, checkpoint in checkpoints.items():
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
    loaded_models[name] = (tokenizer, model)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: textattack/roberta-base-SST-2
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/27 [00:00<?, ?it/s]

In [3]:
test_texts = [
    "The movie was terrible!",
    "The movie was awesome!",
    "The movie was moderate."
]

for text in test_texts:
  print("=" * 55)
  print("VULNERABILITY 1: UNMASKED PADDING DRIFT")
  print(f"Base Text: '{text}'")
  print("=" * 55)

  for name, (tokenizer, model) in loaded_models.items():
      # Scenario A: Clean baseline
      clean_inputs = tokenizer(text, return_tensors="pt")

      clean_logits = model(**clean_inputs).logits
      clean_probs = torch.softmax(clean_logits, dim=-1)[0]

      # Scenario B: Padded to 32 tokens, but attention_mask forced to all 1s (Unmasked)
      padded_inputs = tokenizer(text, padding="max_length", max_length=32, return_tensors="pt")

      unmasked_inputs = {
          "input_ids": padded_inputs["input_ids"],
          "attention_mask": torch.ones_like(padded_inputs["input_ids"])  # Ignoring pad mask!
      }

      unmasked_logits = model(**unmasked_inputs).logits
      unmasked_probs = torch.softmax(unmasked_logits, dim=-1)[0]

      # Print clean comparison
      print(f"\n--- {name} ---")
      print(f"Clean Input        -> Neg: {clean_probs[0]*100:.2f}% | Pos: {clean_probs[1]*100:.2f}%")
      print(f"Unmasked Pad Tokens-> Neg: {unmasked_probs[0]*100:.2f}% | Pos: {unmasked_probs[1]*100:.2f}%")

VULNERABILITY 1: UNMASKED PADDING DRIFT
Base Text: 'The movie was terrible!'

--- DistilBERT (WordPiece) ---
Clean Input        -> Neg: 99.97% | Pos: 0.03%
Unmasked Pad Tokens-> Neg: 99.64% | Pos: 0.36%

--- RoBERTa (BPE) ---
Clean Input        -> Neg: 99.90% | Pos: 0.10%
Unmasked Pad Tokens-> Neg: 99.18% | Pos: 0.82%

--- ALBERT (SentencePiece) ---
Clean Input        -> Neg: 99.80% | Pos: 0.20%
Unmasked Pad Tokens-> Neg: 55.96% | Pos: 44.04%
VULNERABILITY 1: UNMASKED PADDING DRIFT
Base Text: 'The movie was awesome!'

--- DistilBERT (WordPiece) ---
Clean Input        -> Neg: 0.01% | Pos: 99.99%
Unmasked Pad Tokens-> Neg: 0.13% | Pos: 99.87%

--- RoBERTa (BPE) ---
Clean Input        -> Neg: 0.03% | Pos: 99.97%
Unmasked Pad Tokens-> Neg: 0.12% | Pos: 99.88%

--- ALBERT (SentencePiece) ---
Clean Input        -> Neg: 0.05% | Pos: 99.95%
Unmasked Pad Tokens-> Neg: 55.94% | Pos: 44.06%
VULNERABILITY 1: UNMASKED PADDING DRIFT
Base Text: 'The movie was moderate.'

--- DistilBERT (WordPiece) --